# 📊 Notebook 01 — Exploratory Data Analysis
**CT Guanabara Futevolei | Data Science Performance Project**

This notebook explores the raw datasets collected from 5 pilot athletes at CT Guanabara,
covering pre-intervention metrics, athlete profiles, and weekly training logs.


In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from data_pipeline import load_raw, compute_ipg

# Load all datasets
raw = load_raw()
profile = raw['profile']
pre     = raw['pre']
post    = raw['post']
log     = raw['log']

print("✅ Datasets loaded successfully")
for k, df in raw.items():
    print(f"  [{k}]: {df.shape[0]} rows × {df.shape[1]} cols")


## 1. Athlete Profiles

In [ ]:
profile.set_index('athlete_id')[[
    'name','age','experience_years','position','level','bmi'
]].style.background_gradient(cmap='Greens', subset=['bmi','experience_years'])


## 2. Pre-Intervention Metrics — Week 1 Baseline

In [ ]:
pre_w1 = pre[pre['week'] == 1].copy()

metrics = ['jump_height_cm','ball_control_touches',
           'serve_accuracy_pct','reception_efficiency_pct','attack_rate_pct']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
colors = ['#1B4332','#2D6A4F','#52B788','#74C69D','#95D5B2']

for ax, col, color in zip(axes, metrics, colors):
    ax.bar(pre_w1['name'].str.split().str[0], pre_w1[col], color=color, edgecolor='white')
    ax.set_title(col.replace('_',' ').title(), fontsize=10, fontweight='bold')
    ax.set_xticklabels(pre_w1['name'].str.split().str[0], rotation=30, ha='right', fontsize=8)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Baseline Metrics — Week 1 (Pre-Intervention)', fontsize=14,
             fontweight='bold', color='#1B4332', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/baseline_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: baseline_metrics.png")


## 3. Distribution of Each Metric

In [ ]:
pre_all = compute_ipg(pre)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(metrics):
    sns.histplot(data=pre_all, x=col, kde=True, ax=axes[i],
                 color='#2D6A4F', edgecolor='white', alpha=0.8)
    axes[i].set_title(col.replace('_',' ').title(), fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].spines[['top','right']].set_visible(False)

# IPG distribution
sns.histplot(data=pre_all, x='ipg_computed', kde=True, ax=axes[5],
             color='#E76F51', edgecolor='white', alpha=0.8)
axes[5].set_title('IPG (General Performance Index)', fontweight='bold')
axes[5].spines[['top','right']].set_visible(False)

plt.suptitle('Metric Distributions — Pre-Intervention Phase',
             fontsize=14, fontweight='bold', color='#1B4332')
plt.tight_layout()
plt.savefig('../outputs/figures/distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: distributions.png")


## 4. Correlation Heatmap

In [ ]:
corr_cols = metrics + ['ipg']
corr = pre[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(145, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 11})
ax.set_title('Correlation Matrix — Pre-Intervention Metrics',
             fontsize=13, fontweight='bold', color='#1B4332', pad=15)
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: correlation_heatmap.png")


## 5. Sessions per Week vs Ball Control (Key Correlation)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors_ath = ['#1B4332','#E76F51','#2D6A4F','#74C69D','#95D5B2']

for (aid, group), color in zip(pre.groupby('athlete_id'), colors_ath):
    name = group['name'].iloc[0].split()[0]
    ax.scatter(group['sessions_per_week'], group['ball_control_touches'],
               label=name, color=color, s=80, alpha=0.8, edgecolors='white')

# Trend line
x = pre['sessions_per_week'].values
y = pre['ball_control_touches'].values
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
xline = np.linspace(x.min(), x.max(), 100)
ax.plot(xline, p(xline), '--', color='#6C757D', lw=1.5, label='Trend')

r = np.corrcoef(x, y)[0,1]
ax.set_xlabel('Sessions per Week', fontsize=12)
ax.set_ylabel('Ball Control (touches)', fontsize=12)
ax.set_title(f'Training Frequency × Ball Control  |  r = {r:.2f}',
             fontsize=13, fontweight='bold', color='#1B4332')
ax.legend(frameon=False)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/figures/sessions_vs_ball_control.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Correlation r = {r:.2f}")


## 6. Athlete Summary Statistics

In [ ]:
summary = pre.groupby('name')[metrics + ['ipg']].agg(['mean','std']).round(2)
summary.columns = [f'{m}_{s}' for m,s in summary.columns]
print("Summary Statistics — Pre-Intervention Phase")
display(summary)
summary.to_csv('../outputs/pre_summary_stats.csv')
print("✅ Saved: pre_summary_stats.csv")
